# 09 -- Historical Odds Scraper

Scrapes moneyline odds from BestFightOdds for 10 numbered UFC events.
Averages odds across all available sportsbooks per fighter.

| Cell | Stage | Output |
|------|-------|--------|
| 1 | Setup | session, helpers |
| 2 | Event URLs | hardcoded list |
| 3 | Scrape + parse | all 10 events |
| 4 | Average + clean | compute avg odds, add vig-free probs |
| 5 | Save + summary | `data/odds_historical.csv` |

**BFO page structure:**
- Two tables: Table 0 = left headers (names), Table 1 = odds data
- Fighter names in `<span class='t-b-fcc'>`
- Moneyline odds in `<td class='but-sg'>` (skip `but-sgp` = props)
- Prop rows have `class='pr'` -- skip
- Fights come in row pairs: fighter 1 then fighter 2
- Odds text has arrows to strip: `+207\u25b2`, `-255\u25bc`

In [22]:
# Cell 1: Imports & Setup

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import os
import time
from tqdm import tqdm

BASE = 'https://www.bestfightodds.com'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0.0.0 Safari/537.36',
    'Referer': 'https://www.bestfightodds.com/',
}

DELAY = 2.0
DATA_DIR = './data' if os.path.exists('./data/model_data.csv') else '../data'
os.makedirs(DATA_DIR, exist_ok=True)

session = requests.Session()
session.headers.update(HEADERS)

def get_soup(url):
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    time.sleep(DELAY)
    return BeautifulSoup(resp.text, 'lxml')

print(f'Ready  |  delay={DELAY}s  |  data_dir={DATA_DIR}')

Ready  |  delay=2.0s  |  data_dir=./data


In [23]:
# Cell 2: Event URLs (hardcoded)

EVENT_URLS = [
    'https://www.bestfightodds.com/events/ufc-315-3708',
    'https://www.bestfightodds.com/events/ufc-316-3702',
    'https://www.bestfightodds.com/events/ufc-319-3800',
    'https://www.bestfightodds.com/events/ufc-320-3853',
    'https://www.bestfightodds.com/events/ufc-321-odds-3780',
    'https://www.bestfightodds.com/events/ufc-322-3830',
    'https://www.bestfightodds.com/events/ufc-323-3951',
    'https://www.bestfightodds.com/events/ufc-324-3973',
    'https://www.bestfightodds.com/events/ufc-326-4065',
    'https://www.bestfightodds.com/events/ufc-327-4074',
]

print(f'{len(EVENT_URLS)} events to scrape')
for url in EVENT_URLS:
    print(f'  {url}')

10 events to scrape
  https://www.bestfightodds.com/events/ufc-315-3708
  https://www.bestfightodds.com/events/ufc-316-3702
  https://www.bestfightodds.com/events/ufc-319-3800
  https://www.bestfightodds.com/events/ufc-320-3853
  https://www.bestfightodds.com/events/ufc-321-odds-3780
  https://www.bestfightodds.com/events/ufc-322-3830
  https://www.bestfightodds.com/events/ufc-323-3951
  https://www.bestfightodds.com/events/ufc-324-3973
  https://www.bestfightodds.com/events/ufc-326-4065
  https://www.bestfightodds.com/events/ufc-327-4074


In [24]:
# Cell 3: Scrape odds from each event page
#
# Page structure:
#   Table 0 (odds-table-responsive-header): left column with fighter names
#   Table 1 (odds-table): odds data with sportsbook columns
#
#   Table 0 rows:
#     - Fighter 1 row: has id='mu-XXXXX', name in <span class='t-b-fcc'>
#     - Fighter 2 row: name in <span class='t-b-fcc'>, no id
#     - Prop rows: class='pr' -- Over/Under, method, round lines
#
#   Table 1 rows (same order):
#     - Fighter 1: odds in <td class='but-sg'> cells
#     - Fighter 2: odds in <td class='but-sg'> cells
#     - Prop rows: <td class='but-sgp'> -- skip
#
#   Odds text: '+207\u25b2', '-255\u25bc' -- strip arrows

def parse_odds_text(text):
    # Strip arrows and unicode, extract American odds integer
    text = text.strip()
    # remove common arrow chars
    text = text.replace('\u25b2', '').replace('\u25bc', '')
    text = text.replace('\u2191', '').replace('\u2193', '')
    text = text.replace('\u2212', '-')  # unicode minus
    text = text.strip()
    if not text or text in ('', '-', 'N/A'):
        return np.nan
    m = re.search(r'([+-]?\d+)', text)
    if m:
        return int(m.group(1))
    return np.nan


def scrape_event(url):
    soup = get_soup(url)

    h1 = soup.select_one('h1')
    event_name = h1.get_text(strip=True) if h1 else ''

    date_el = soup.select_one('span.table-header-date')
    date_text = date_el.get_text(strip=True) if date_el else ''

    tables = soup.select('table.odds-table')
    if len(tables) < 2:
        print(f'  WARNING: {event_name} -- less than 2 tables')
        return []

    name_rows = tables[0].select('tr')
    odds_rows = tables[1].select('tr')

    # collect non-prop fighter rows (indices)
    fighter_indices = []
    for i, nr in enumerate(name_rows):
        if 'pr' in nr.get('class', []):
            continue
        if nr.select_one('span.t-b-fcc'):
            fighter_indices.append(i)

    # pair them up: [0,1], [2,3], [4,5], ...
    fights = []
    for p in range(0, len(fighter_indices) - 1, 2):
        idx1 = fighter_indices[p]
        idx2 = fighter_indices[p + 1]

        name1 = name_rows[idx1].select_one('span.t-b-fcc').get_text(strip=True)
        name2 = name_rows[idx2].select_one('span.t-b-fcc').get_text(strip=True)

        odds1_list = []
        odds2_list = []

        if idx1 < len(odds_rows):
            for td in odds_rows[idx1].select('td.but-sg'):
                val = parse_odds_text(td.get_text(strip=True))
                if not np.isnan(val):
                    odds1_list.append(int(val))

        if idx2 < len(odds_rows):
            for td in odds_rows[idx2].select('td.but-sg'):
                val = parse_odds_text(td.get_text(strip=True))
                if not np.isnan(val):
                    odds2_list.append(int(val))

        if odds1_list or odds2_list:
            fights.append({
                'event_name': event_name,
                'event_date': date_text,
                'event_url': url,
                'fighter_1': name1,
                'fighter_2': name2,
                'odds_1_all': odds1_list,
                'odds_2_all': odds2_list,
                'n_books_1': len(odds1_list),
                'n_books_2': len(odds2_list),
            })

    return fights

# re-scrape
all_fights = []
for url in tqdm(EVENT_URLS, desc='Events'):
    try:
        fights = scrape_event(url)
        all_fights.extend(fights)
        print(f'  {fights[0]["event_name"] if fights else "?"}: {len(fights)} fights')
    except Exception as e:
        print(f'  ERROR {url}: {e}')

print(f'\nTotal: {len(all_fights)} fights')

# quick check
for f in all_fights[:5]:
    print(f"  {f['fighter_1']:25s} vs {f['fighter_2']:25s}  "
          f"odds1={f['odds_1_all'][:3]}...({f['n_books_1']})  "
          f"odds2={f['odds_2_all'][:3]}...({f['n_books_2']})")

Events:  10%|█         | 1/10 [00:02<00:21,  2.39s/it]

  UFC 315: 3 fights


Events:  20%|██        | 2/10 [00:05<00:20,  2.54s/it]

  UFC 316: 5 fights


Events:  30%|███       | 3/10 [00:08<00:19,  2.81s/it]

...

  UFC 324: 3 fights


Events:  90%|█████████ | 9/10 [00:24<00:02,  2.57s/it]

  UFC 326: 3 fights


Events: 100%|██████████| 10/10 [00:27<00:00,  2.73s/it]

  UFC 327: 12 fights

Total: 67 fights
  Bruno Silva               vs Marc Andre Barriault       odds1=[136, 145, 133]...(5)  odds2=[-162, -175, -165]...(5)
  Daniel Santos             vs Jeong Yeong Lee            odds1=[114, 120, 114]...(5)  odds2=[-134, -145, -139]...(5)
  Bekzat Almakhan           vs Brad Katona                odds1=[-142, -145, -159]...(5)  odds2=[120, 122, 128]...(5)
  Ariane da Silva           vs Wang Cong                  odds1=[460, 350, 425]...(3)  odds2=[-650, -500, -575]...(3)
  Ariane Lipski             vs Cong Wang                  odds1=[430, 420]...(2)  odds2=[-590, -625]...(2)


In [25]:
# Cell 4: Compute average and best odds, add derived columns

odds_df = pd.DataFrame(all_fights)
print(f'Raw records: {len(odds_df)}')

# average odds across books
odds_df['odds_1_avg'] = odds_df['odds_1_all'].apply(
    lambda x: int(round(np.mean(x))) if len(x) > 0 else np.nan)
odds_df['odds_2_avg'] = odds_df['odds_2_all'].apply(
    lambda x: int(round(np.mean(x))) if len(x) > 0 else np.nan)

# best odds (most favorable to bettor)
odds_df['odds_1_best'] = odds_df['odds_1_all'].apply(
    lambda x: int(max(x)) if len(x) > 0 else np.nan)
odds_df['odds_2_best'] = odds_df['odds_2_all'].apply(
    lambda x: int(max(x)) if len(x) > 0 else np.nan)

# implied probabilities (from average odds)
def american_to_implied(am):
    if pd.isna(am):
        return np.nan
    am = int(am)
    return 100 / (am + 100) if am > 0 else abs(am) / (abs(am) + 100)

odds_df['imp_1'] = odds_df['odds_1_avg'].apply(american_to_implied)
odds_df['imp_2'] = odds_df['odds_2_avg'].apply(american_to_implied)
odds_df['overround'] = odds_df['imp_1'] + odds_df['imp_2']

# vig-free probabilities
odds_df['fair_1'] = odds_df['imp_1'] / odds_df['overround']
odds_df['fair_2'] = odds_df['imp_2'] / odds_df['overround']

# drop rows with no odds on either side
has_both = odds_df['odds_1_avg'].notna() & odds_df['odds_2_avg'].notna()
dropped = (~has_both).sum()
odds_df = odds_df[has_both].reset_index(drop=True)
print(f'With both sides: {len(odds_df)} (dropped {dropped})')

# summary
print(f'\nEvents: {odds_df["event_name"].nunique()}')
print(f'Avg books per fighter: {odds_df["n_books_1"].mean():.1f}')
print(f'Avg overround: {odds_df["overround"].mean():.3f}')

print()
for ev in odds_df['event_name'].unique():
    sub = odds_df[odds_df['event_name'] == ev]
    print(f'  {ev:30s}  {len(sub):2d} fights  avg vig {sub["overround"].mean():.3f}')

print()
print(odds_df[['event_name','fighter_1','fighter_2',
               'odds_1_avg','odds_2_avg','n_books_1','fair_1','fair_2']]
      .head(15).to_string(index=False))

Raw records: 67
With both sides: 67 (dropped 0)

Events: 10
Avg books per fighter: 5.9
Avg overround: 1.057

  UFC 315                          3 fights  avg vig 1.047
  UFC 316                          5 fights  avg vig 1.048
  UFC 319                          9 fights  avg vig 1.048
  UFC 320                          5 fights  avg vig 1.047
  UFC 321 Odds                    13 fights  avg vig 1.075
  UFC 322                         12 fights  avg vig 1.041
  UFC 323                          2 fights  avg vig 1.045
  UFC 324                          3 fights  avg vig 1.009
  UFC 326                          3 fights  avg vig 1.049
  UFC 327                         12 fights  avg vig 1.087

event_name           fighter_1            fighter_2  odds_1_avg  odds_2_avg  n_books_1   fair_1   fair_2
   UFC 315         Bruno Silva Marc Andre Barriault         137        -167          5 0.402843 0.597157
   UFC 315       Daniel Santos      Jeong Yeong Lee         113        -137          5 0.4

In [26]:
# Cell 5: Save odds_historical.csv

save_cols = ['event_name', 'event_date', 'event_url',
             'fighter_1', 'fighter_2',
             'odds_1_avg', 'odds_2_avg',
             'odds_1_best', 'odds_2_best',
             'n_books_1', 'n_books_2',
             'imp_1', 'imp_2', 'overround',
             'fair_1', 'fair_2']

out = f'{DATA_DIR}/odds_historical.csv'
odds_df[save_cols].to_csv(out, index=False)

print(f'Saved {len(odds_df)} fights to {out}')
print(f'  Events: {odds_df["event_name"].nunique()}')
print(f'  Fights with both sides: {len(odds_df)}')
print(f'  File size: {os.path.getsize(out)/1024:.1f} KB')
print(f'\nNext: notebook 10 (backtest) merges these with model predictions')

Saved 67 fights to ./data/odds_historical.csv
  Events: 10
  Fights with both sides: 67
  File size: 14.4 KB

Next: notebook 10 (backtest) merges these with model predictions


In [27]:
# Debug: check what we actually scraped
for f in all_fights[:5]:
    print(f"  {f['fighter_1']:25s} vs {f['fighter_2']:25s}  odds1={f['odds_1_all']}  odds2={f['odds_2_all']}")

  Bruno Silva               vs Marc Andre Barriault       odds1=[136, 145, 133, 130, 142]  odds2=[-162, -175, -165, -163, -170]
  Daniel Santos             vs Jeong Yeong Lee            odds1=[114, 120, 114, 105, 114]  odds2=[-134, -145, -139, -130, -135]
  Bekzat Almakhan           vs Brad Katona                odds1=[-142, -145, -159, -163, -142]  odds2=[120, 122, 128, 130, 120]
  Ariane da Silva           vs Wang Cong                  odds1=[460, 350, 425]  odds2=[-650, -500, -575]
  Ariane Lipski             vs Cong Wang                  odds1=[430, 420]  odds2=[-590, -625]


In [28]:
# After scraping each event, print what was found vs what was expected
for event_url in event_urls:
    # ... scrape ...
    print(f"{event_name}: {len(fights_found)} fights parsed from {len(all_rows)} table rows")

NameError: name 'event_urls' is not defined